In [ ]:
# ============================================================
# DENGUE HOST–VIRUS DYNAMICS PROJECT
# NOTEBOOK 02 — PRE-LANDMARK TRAJECTORIES
# ============================================================
#
# Purpose:
# Construct all predictors using information available
# on or before DOI 4 only.
#
# Primary landmark:
# DOI 4
#
# Important rule:
# NO DOI > 4 information may enter predictor construction.
#
# ============================================================

from pathlib import Path
import sys
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

PROJECT_NAME = "Dengue Host–Virus Dynamics"
NOTEBOOK = "02_PreLandmark_Trajectories"
RANDOM_SEED = 10010
PRIMARY_LANDMARK = 4

np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT /
    "data" /
    "Longitudinal viremia and platelet 22May2024.csv"
)

COHORT_PATH = (
    PROJECT_ROOT /
    "outputs" /
    "audit" /
    "DOI4_Patient_Level_Cohort_Definition.csv"
)

print("=" * 70)
print(PROJECT_NAME)
print("=" * 70)

print("Notebook          :", NOTEBOOK)
print("Primary landmark  :", PRIMARY_LANDMARK)
print("Random seed       :", RANDOM_SEED)

print("\nRaw dataset path:")
print(DATA_PATH)

print("\nFrozen cohort path:")
print(COHORT_PATH)

print("\nFiles found:")
print("Raw dataset :", DATA_PATH.exists())
print("DOI-4 cohort:", COHORT_PATH.exists())

assert DATA_PATH.exists(), "Raw dengue dataset not found."
assert COHORT_PATH.exists(), "Frozen DOI-4 cohort file not found."

print("\nNotebook 02 initialized successfully.")

In [ ]:
print("=" * 70)
print("LOAD DATA AND CREATE STRICT DOI <= 4 DATASET")
print("=" * 70)

# ============================================================
# LOAD RAW LONGITUDINAL DATA
# ============================================================

df_raw = pd.read_csv(DATA_PATH)

# Safe patient identifier
df_raw["Patient_ID"] = (
    df_raw["Study"].astype(str).str.strip()
    + "__"
    + df_raw["Code"].astype(str).str.strip()
)

# Convert DOI safely
df_raw["DOI_num"] = pd.to_numeric(
    df_raw["DOI"],
    errors="coerce"
)

# ============================================================
# LOAD FROZEN DOI-4 PATIENT-LEVEL COHORT
# ============================================================

cohort_doi4 = pd.read_csv(COHORT_PATH)

# ============================================================
# CREATE STRICT PRE-LANDMARK DATASET
# ============================================================

df_pre4 = (
    df_raw.loc[
        df_raw["DOI_num"] <= PRIMARY_LANDMARK
    ]
    .copy()
)

# ============================================================
# BASIC CHECKS
# ============================================================

print("Raw longitudinal rows :", len(df_raw))
print("Raw patients          :", df_raw["Patient_ID"].nunique())

print("\nDOI <= 4 rows         :", len(df_pre4))
print("DOI <= 4 patients     :", df_pre4["Patient_ID"].nunique())

print("\nMinimum DOI in pre-landmark data:",
      df_pre4["DOI_num"].min())

print("Maximum DOI in pre-landmark data:",
      df_pre4["DOI_num"].max())

# Critical leakage-prevention assertion
assert df_pre4["DOI_num"].max() <= PRIMARY_LANDMARK

# Confirm expected DOI-4 observed cohort
assert df_pre4["Patient_ID"].nunique() == 2390

print("\nTemporal leakage check passed:")
print("No DOI > 4 observations are present.")

print("\nFrozen patient-level cohort rows:",
      len(cohort_doi4))

print("\nCell 2 complete.")

In [ ]:
print("=" * 70)
print("PRE-LANDMARK MEASUREMENT DEPTH AUDIT")
print("=" * 70)

# ============================================================
# PATIENT-LEVEL COUNTS OF AVAILABLE MEASUREMENTS BY DOI 4
# ============================================================

measurement_depth = (
    df_pre4.groupby("Patient_ID")
           .agg(
               N_rows=("DOI_num", "size"),

               N_PLT=(
                   "PLT",
                   lambda x: x.notna().sum()
               ),

               N_VIR=(
                   "vir",
                   lambda x: x.notna().sum()
               ),

               First_DOI=("DOI_num", "min"),
               Last_DOI=("DOI_num", "max")
           )
           .reset_index()
)

print("Patients:", len(measurement_depth))

# ============================================================
# SUMMARY DISTRIBUTION
# ============================================================

print("\nMeasurement count summary:")
display(
    measurement_depth[
        [
            "N_rows",
            "N_PLT",
            "N_VIR",
            "First_DOI",
            "Last_DOI"
        ]
    ].describe()
)

# ============================================================
# JOINT AVAILABILITY
# ============================================================

availability_categories = pd.DataFrame({
    "Criterion": [
        ">=1 platelet and >=1 viremia",
        ">=2 platelet and >=2 viremia",
        ">=3 platelet and >=3 viremia",
        ">=4 platelet and >=4 viremia"
    ],
    "Patients": [
        (
            (measurement_depth["N_PLT"] >= 1) &
            (measurement_depth["N_VIR"] >= 1)
        ).sum(),

        (
            (measurement_depth["N_PLT"] >= 2) &
            (measurement_depth["N_VIR"] >= 2)
        ).sum(),

        (
            (measurement_depth["N_PLT"] >= 3) &
            (measurement_depth["N_VIR"] >= 3)
        ).sum(),

        (
            (measurement_depth["N_PLT"] >= 4) &
            (measurement_depth["N_VIR"] >= 4)
        ).sum()
    ]
})

availability_categories["Percent"] = (
    100 *
    availability_categories["Patients"] /
    len(measurement_depth)
).round(2)

print("\nJoint platelet-viremia availability:")
display(availability_categories)

# ============================================================
# EXACT DOI-SPECIFIC AVAILABILITY
# ============================================================

doi_availability = []

for doi in [1, 2, 3, 4]:

    temp = df_pre4[
        df_pre4["DOI_num"] == doi
    ]

    doi_availability.append({
        "DOI": doi,
        "Patients": temp["Patient_ID"].nunique(),

        "PLT_available":
            temp.loc[temp["PLT"].notna(),
                     "Patient_ID"].nunique(),

        "Viremia_available":
            temp.loc[temp["vir"].notna(),
                     "Patient_ID"].nunique(),

        "Both_available":
            temp.loc[
                temp["PLT"].notna() &
                temp["vir"].notna(),
                "Patient_ID"
            ].nunique()
    })

doi_availability = pd.DataFrame(doi_availability)

for col in [
    "PLT_available",
    "Viremia_available",
    "Both_available"
]:
    doi_availability[col + "_%"] = (
        100 *
        doi_availability[col] /
        doi_availability["Patients"]
    ).round(2)

print("\nDOI-specific measurement availability:")
display(doi_availability)

# ============================================================
# COMPLETE DOI 3 + DOI 4 PAIRS
# Important for early slope calculations
# ============================================================

doi34 = (
    df_pre4[
        df_pre4["DOI_num"].isin([3, 4])
    ]
    .pivot_table(
        index="Patient_ID",
        columns="DOI_num",
        values=["PLT", "vir"],
        aggfunc="first"
    )
)

complete_34 = (
    doi34[("PLT", 3)].notna() &
    doi34[("PLT", 4)].notna() &
    doi34[("vir", 3)].notna() &
    doi34[("vir", 4)].notna()
)

print("\nPatients with complete DOI 3 -> DOI 4")
print("platelet + viremia pairs:",
      int(complete_34.sum()))

print(
    "Percent:",
    round(
        100 * complete_34.mean(),
        2
    ),
    "%"
)

print("\nCell 3 complete.")

In [ ]:
print("=" * 70)
print("DOI 3 -> DOI 4 PAIRED DYNAMIC FEATURE CONSTRUCTION")
print("=" * 70)

# ============================================================
# 1. Extract DOI 3 and DOI 4 measurements
# ============================================================

doi34_long = (
    df_pre4.loc[
        df_pre4["DOI_num"].isin([3, 4]),
        [
            "Patient_ID",
            "DOI_num",
            "PLT",
            "vir"
        ]
    ]
    .copy()
)

# Ensure numeric
doi34_long["PLT"] = pd.to_numeric(
    doi34_long["PLT"],
    errors="coerce"
)

doi34_long["vir"] = pd.to_numeric(
    doi34_long["vir"],
    errors="coerce"
)

# ============================================================
# 2. Log-transform viral burden
# log10(viremia + 1) safely handles zero viral load
# ============================================================

doi34_long["log10_vir_plus1"] = np.log10(
    doi34_long["vir"] + 1
)

# ============================================================
# 3. Convert to one row per patient
# ============================================================

paired = (
    doi34_long
    .pivot(
        index="Patient_ID",
        columns="DOI_num",
        values=[
            "PLT",
            "vir",
            "log10_vir_plus1"
        ]
    )
)

# Flatten column names
paired.columns = [
    f"{variable}_DOI{int(doi)}"
    for variable, doi in paired.columns
]

paired = paired.reset_index()

# ============================================================
# 4. Rename log viral variables for readability
# ============================================================

paired = paired.rename(
    columns={
        "log10_vir_plus1_DOI3": "LogVIR_DOI3",
        "log10_vir_plus1_DOI4": "LogVIR_DOI4"
    }
)

# ============================================================
# 5. Define complete paired cohort
# ============================================================

required_pair_vars = [
    "PLT_DOI3",
    "PLT_DOI4",
    "vir_DOI3",
    "vir_DOI4"
]

paired["Complete_D34_Pair"] = (
    paired[required_pair_vars]
    .notna()
    .all(axis=1)
)

# ============================================================
# 6. Calculate early dynamic features
# ============================================================

# Platelet change:
# Negative = platelet decline
# Positive = platelet recovery/rise
paired["Delta_PLT_D3_D4"] = (
    paired["PLT_DOI4"] -
    paired["PLT_DOI3"]
)

# Viral change on raw scale
paired["Delta_VIR_D3_D4"] = (
    paired["vir_DOI4"] -
    paired["vir_DOI3"]
)

# Viral change on log10 scale
# Negative = viral burden falling
# Positive = viral burden rising
paired["Delta_LogVIR_D3_D4"] = (
    paired["LogVIR_DOI4"] -
    paired["LogVIR_DOI3"]
)

# ============================================================
# 7. Directional state indicators
# ============================================================

paired["Virus_Falling_D3_D4"] = (
    paired["Delta_LogVIR_D3_D4"] < 0
)

paired["Platelets_Falling_D3_D4"] = (
    paired["Delta_PLT_D3_D4"] < 0
)

# Important biological state:
# Virus falling BUT platelets still falling
paired["Discordant_Recovery_D3_D4"] = (
    paired["Complete_D34_Pair"]
    &
    paired["Virus_Falling_D3_D4"]
    &
    paired["Platelets_Falling_D3_D4"]
)

# Both biological systems moving toward recovery:
# Virus falling and platelets stable/rising
paired["Concordant_Recovery_D3_D4"] = (
    paired["Complete_D34_Pair"]
    &
    paired["Virus_Falling_D3_D4"]
    &
    ~paired["Platelets_Falling_D3_D4"]
)

# ============================================================
# 8. Restrict primary paired feature dataset
# ============================================================

features_d34 = (
    paired.loc[
        paired["Complete_D34_Pair"]
    ]
    .copy()
)

# ============================================================
# 9. Integrity checks
# ============================================================

print("Patients with DOI 3 or DOI 4 records:",
      len(paired))

print(
    "Complete DOI 3 -> DOI 4 paired cohort:",
    len(features_d34)
)

assert len(features_d34) == 2128

print("\nMissing values in required paired variables:")
print(
    features_d34[required_pair_vars]
    .isna()
    .sum()
)

# ============================================================
# 10. Dynamic feature summary
# ============================================================

summary_vars = [
    "PLT_DOI3",
    "PLT_DOI4",
    "Delta_PLT_D3_D4",
    "vir_DOI3",
    "vir_DOI4",
    "LogVIR_DOI3",
    "LogVIR_DOI4",
    "Delta_LogVIR_D3_D4"
]

print("\nDynamic feature distributions:")
display(
    features_d34[summary_vars]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75
        ]
    )
    .T
)

# ============================================================
# 11. Directional biological states
# ============================================================

state_summary = pd.DataFrame({
    "State": [
        "Viral burden falling",
        "Platelets falling",
        "Virus falling + platelets falling",
        "Virus falling + platelets stable/rising"
    ],

    "Patients": [
        features_d34[
            "Virus_Falling_D3_D4"
        ].sum(),

        features_d34[
            "Platelets_Falling_D3_D4"
        ].sum(),

        features_d34[
            "Discordant_Recovery_D3_D4"
        ].sum(),

        features_d34[
            "Concordant_Recovery_D3_D4"
        ].sum()
    ]
})

state_summary["Percent"] = (
    100 *
    state_summary["Patients"] /
    len(features_d34)
).round(2)

print("\nDOI 3 -> DOI 4 directional states:")
display(state_summary)

print("\nCell 4 complete.")

In [ ]:
print("=" * 70)
print("ATTACH FROZEN POST-DOI-4 OUTCOMES")
print("=" * 70)

# ============================================================
# 1. Select outcome information from frozen DOI-4 cohort
# ============================================================

outcome_columns = [
    "Patient_ID",
    "Age",
    "Sex",
    "Study",
    "Serotype",
    "Severe_dengue",
    "Leakage",
    "Eligible_severe_prediction",
    "Severe_future_outcome",
    "Eligible_leakage_prediction",
    "Leakage_future_outcome"
]

outcomes_doi4 = cohort_doi4[
    outcome_columns
].copy()

# ============================================================
# 2. Merge DOI 3-4 predictors with frozen outcomes
# ============================================================

analysis_d34 = features_d34.merge(
    outcomes_doi4,
    on="Patient_ID",
    how="left",
    validate="one_to_one"
)

print("Complete DOI 3->4 feature cohort:",
      len(features_d34))

print("Merged analysis cohort:",
      len(analysis_d34))

assert len(analysis_d34) == 2128

# ============================================================
# 3. Check outcome linkage
# ============================================================

print("\nPatients missing severe eligibility information:")
print(
    analysis_d34[
        "Eligible_severe_prediction"
    ].isna().sum()
)

print("\nPatients missing leakage eligibility information:")
print(
    analysis_d34[
        "Eligible_leakage_prediction"
    ].isna().sum()
)

# ============================================================
# 4. Create separate outcome-specific datasets
# ============================================================

severe_d34 = analysis_d34.loc[
    analysis_d34["Eligible_severe_prediction"] == True
].copy()

leakage_d34 = analysis_d34.loc[
    analysis_d34["Eligible_leakage_prediction"] == True
].copy()

# ============================================================
# 5. Ensure outcomes are numeric
# ============================================================

severe_d34["Severe_future_outcome"] = pd.to_numeric(
    severe_d34["Severe_future_outcome"],
    errors="coerce"
)

leakage_d34["Leakage_future_outcome"] = pd.to_numeric(
    leakage_d34["Leakage_future_outcome"],
    errors="coerce"
)

# ============================================================
# 6. Cohort summaries
# ============================================================

print("\n" + "-" * 70)
print("SEVERE DENGUE DOI 3->4 FEATURE COHORT")
print("-" * 70)

print("Eligible patients :", len(severe_d34))
print(
    "Future severe events:",
    int(severe_d34["Severe_future_outcome"].sum())
)

print(
    "Event rate:",
    round(
        100 *
        severe_d34["Severe_future_outcome"].mean(),
        2
    ),
    "%"
)

print("\n" + "-" * 70)
print("PLASMA LEAKAGE DOI 3->4 FEATURE COHORT")
print("-" * 70)

print("Eligible patients :", len(leakage_d34))
print(
    "Future leakage events:",
    int(leakage_d34["Leakage_future_outcome"].sum())
)

print(
    "Event rate:",
    round(
        100 *
        leakage_d34["Leakage_future_outcome"].mean(),
        2
    ),
    "%"
)

# ============================================================
# 7. Confirm no outcome missingness inside analysis cohorts
# ============================================================

assert severe_d34["Severe_future_outcome"].isna().sum() == 0
assert leakage_d34["Leakage_future_outcome"].isna().sum() == 0

print("\nOutcome linkage integrity checks passed.")
print("Cell 5 complete.")

In [ ]:
print("=" * 70)
print("EARLY HOST-VIRUS DISCORDANCE VS FUTURE OUTCOMES")
print("=" * 70)

from scipy.stats import fisher_exact
from scipy.stats import norm

# ============================================================
# FUNCTION FOR TRANSPARENT 2x2 RISK ANALYSIS
# ============================================================

def binary_risk_analysis(data, exposure, outcome, outcome_name):
    
    temp = data[[exposure, outcome]].dropna().copy()

    temp[exposure] = temp[exposure].astype(bool)
    temp[outcome] = temp[outcome].astype(int)

    # --------------------------------------------------------
    # 2x2 counts
    # --------------------------------------------------------

    a = int(
        ((temp[exposure] == True) &
         (temp[outcome] == 1)).sum()
    )

    b = int(
        ((temp[exposure] == True) &
         (temp[outcome] == 0)).sum()
    )

    c = int(
        ((temp[exposure] == False) &
         (temp[outcome] == 1)).sum()
    )

    d = int(
        ((temp[exposure] == False) &
         (temp[outcome] == 0)).sum()
    )

    exposed_n = a + b
    unexposed_n = c + d

    risk_exposed = a / exposed_n
    risk_unexposed = c / unexposed_n

    # --------------------------------------------------------
    # Risk ratio
    # --------------------------------------------------------

    rr = risk_exposed / risk_unexposed

    # Log RR confidence interval
    if a > 0 and c > 0:

        se_log_rr = np.sqrt(
            (1 / a) -
            (1 / exposed_n) +
            (1 / c) -
            (1 / unexposed_n)
        )

        log_rr = np.log(rr)

        rr_lower = np.exp(
            log_rr - 1.96 * se_log_rr
        )

        rr_upper = np.exp(
            log_rr + 1.96 * se_log_rr
        )

    else:

        rr_lower = np.nan
        rr_upper = np.nan

    # --------------------------------------------------------
    # Absolute risk difference
    # --------------------------------------------------------

    risk_difference = (
        risk_exposed -
        risk_unexposed
    )

    # --------------------------------------------------------
    # Fisher exact test
    # --------------------------------------------------------

    odds_ratio, fisher_p = fisher_exact(
        [[a, b],
         [c, d]],
        alternative="two-sided"
    )

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    result = {
        "Outcome": outcome_name,

        "Discordant_N":
            exposed_n,

        "Discordant_Events":
            a,

        "Discordant_Risk_%":
            round(100 * risk_exposed, 2),

        "NonDiscordant_N":
            unexposed_n,

        "NonDiscordant_Events":
            c,

        "NonDiscordant_Risk_%":
            round(100 * risk_unexposed, 2),

        "Risk_Ratio":
            round(rr, 3),

        "RR_95CI_Lower":
            round(rr_lower, 3),

        "RR_95CI_Upper":
            round(rr_upper, 3),

        "Absolute_Risk_Difference_%":
            round(
                100 * risk_difference,
                2
            ),

        "Odds_Ratio":
            round(odds_ratio, 3),

        "Fisher_P":
            fisher_p
    }

    return result


# ============================================================
# SEVERE DENGUE
# ============================================================

severe_result = binary_risk_analysis(
    data=severe_d34,
    exposure="Discordant_Recovery_D3_D4",
    outcome="Severe_future_outcome",
    outcome_name="Future severe dengue"
)

# ============================================================
# PLASMA LEAKAGE
# ============================================================

leakage_result = binary_risk_analysis(
    data=leakage_d34,
    exposure="Discordant_Recovery_D3_D4",
    outcome="Leakage_future_outcome",
    outcome_name="Future plasma leakage"
)

# ============================================================
# COMBINE RESULTS
# ============================================================

discordance_results = pd.DataFrame(
    [
        severe_result,
        leakage_result
    ]
)

print("\nEARLY DISCORDANCE ASSOCIATION RESULTS")
display(discordance_results)

# ============================================================
# HUMAN-READABLE 2x2 TABLES
# ============================================================

print("\n" + "=" * 70)
print("SEVERE DENGUE — EVENT COUNTS")
print("=" * 70)

severe_table = pd.crosstab(
    severe_d34["Discordant_Recovery_D3_D4"],
    severe_d34["Severe_future_outcome"],
    margins=True
)

display(severe_table)

print("\n" + "=" * 70)
print("PLASMA LEAKAGE — EVENT COUNTS")
print("=" * 70)

leakage_table = pd.crosstab(
    leakage_d34["Discordant_Recovery_D3_D4"],
    leakage_d34["Leakage_future_outcome"],
    margins=True
)

display(leakage_table)

# ============================================================
# EXPORT
# ============================================================

TABLE_DIR = (
    PROJECT_ROOT /
    "outputs" /
    "tables"
)

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

discordance_results.to_csv(
    TABLE_DIR /
    "Table_Early_Discordance_Future_Outcomes.csv",
    index=False
)

print("\nSaved:")
print(
    TABLE_DIR /
    "Table_Early_Discordance_Future_Outcomes.csv"
)

print("\nCell 6 complete.")

In [ ]:
print("=" * 70)
print("CONTINUOUS EARLY DYNAMICS VS FUTURE OUTCOMES")
print("=" * 70)

from scipy.stats import mannwhitneyu

# ============================================================
# FEATURES SPECIFIED BEFORE OUTCOME COMPARISON
# ============================================================

continuous_features = [
    "PLT_DOI3",
    "PLT_DOI4",
    "Delta_PLT_D3_D4",
    "LogVIR_DOI3",
    "LogVIR_DOI4",
    "Delta_LogVIR_D3_D4"
]

# ============================================================
# BENJAMINI-HOCHBERG FDR FUNCTION
# ============================================================

def bh_fdr(pvalues):
    p = np.asarray(pvalues, dtype=float)

    order = np.argsort(p)
    ranked = p[order]

    n = len(p)

    adjusted = ranked * n / np.arange(1, n + 1)

    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]

    adjusted = np.clip(adjusted, 0, 1)

    output = np.empty_like(adjusted)
    output[order] = adjusted

    return output


# ============================================================
# ANALYSIS FUNCTION
# ============================================================

def continuous_outcome_comparison(
    data,
    outcome,
    outcome_label
):

    results = []

    for feature in continuous_features:

        temp = data[
            [feature, outcome]
        ].dropna()

        cases = temp.loc[
            temp[outcome] == 1,
            feature
        ]

        controls = temp.loc[
            temp[outcome] == 0,
            feature
        ]

        if len(cases) == 0 or len(controls) == 0:
            continue

        # Mann-Whitney U
        u_stat, p_value = mannwhitneyu(
            cases,
            controls,
            alternative="two-sided"
        )

        # Rank-biserial correlation
        # Positive means larger values in cases
        rank_biserial = (
            2 * u_stat /
            (len(cases) * len(controls))
            - 1
        )

        results.append({
            "Outcome": outcome_label,
            "Feature": feature,

            "Cases_N": len(cases),
            "Controls_N": len(controls),

            "Cases_Median":
                np.median(cases),

            "Cases_Q1":
                np.percentile(cases, 25),

            "Cases_Q3":
                np.percentile(cases, 75),

            "Controls_Median":
                np.median(controls),

            "Controls_Q1":
                np.percentile(controls, 25),

            "Controls_Q3":
                np.percentile(controls, 75),

            "Rank_Biserial_Effect":
                rank_biserial,

            "P_value":
                p_value
        })

    result_df = pd.DataFrame(results)

    result_df["FDR_q"] = bh_fdr(
        result_df["P_value"].values
    )

    return result_df


# ============================================================
# SEVERE DENGUE
# ============================================================

severe_continuous = continuous_outcome_comparison(
    severe_d34,
    "Severe_future_outcome",
    "Future severe dengue"
)

# ============================================================
# PLASMA LEAKAGE
# ============================================================

leakage_continuous = continuous_outcome_comparison(
    leakage_d34,
    "Leakage_future_outcome",
    "Future plasma leakage"
)

# ============================================================
# COMBINE
# ============================================================

continuous_results = pd.concat(
    [
        severe_continuous,
        leakage_continuous
    ],
    ignore_index=True
)

# Global FDR across all tests as additional safeguard
continuous_results["Global_FDR_q"] = bh_fdr(
    continuous_results["P_value"].values
)

print("\nCONTINUOUS FEATURE RESULTS")
display(
    continuous_results.sort_values(
        ["Outcome", "P_value"]
    )
)

# ============================================================
# EXPORT
# ============================================================

continuous_results.to_csv(
    TABLE_DIR /
    "Table_Continuous_Early_Dynamics_Future_Outcomes.csv",
    index=False
)

print("\nSaved:")
print(
    TABLE_DIR /
    "Table_Continuous_Early_Dynamics_Future_Outcomes.csv"
)

print("\nCell 7 complete.")

In [ ]:
# ============================================================
# CELL 8
# CROSS-VALIDATED STATIC VS DYNAMIC MODEL COMPARISON
# DOI 3 -> DOI 4 PRE-LANDMARK FEATURES
# ============================================================

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

print("=" * 70)
print("CROSS-VALIDATED STATIC VS DYNAMIC MODEL COMPARISON")
print("=" * 70)


# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------

SEED = 10010


# ------------------------------------------------------------
# MODEL DEFINITIONS
# ------------------------------------------------------------
# Model A:
#   Demographics only
#
# Model B:
#   Demographics + static DOI-4 platelet/viremia state
#
# Model C:
#   Model B + DOI 3 -> DOI 4 temporal dynamics
#
# IMPORTANT:
# No post-DOI-4 predictors are used.
# ------------------------------------------------------------

MODEL_FEATURES = {

    "A_Demographics": [
        "Age",
        "Sex"
    ],

    "B_Static_DOI4": [
        "Age",
        "Sex",
        "PLT_DOI4",
        "LogVIR_DOI4"
    ],

    "C_Static_plus_Dynamics": [
        "Age",
        "Sex",
        "PLT_DOI4",
        "LogVIR_DOI4",
        "Delta_PLT_D3_D4",
        "Delta_LogVIR_D3_D4"
    ]
}


# ------------------------------------------------------------
# CROSS-VALIDATION FUNCTION
# ------------------------------------------------------------

def evaluate_models_cv(
    data,
    outcome_col,
    outcome_label,
    model_features,
    seed=SEED
):

    results = []

    print("\n" + "-" * 70)
    print(outcome_label.upper())
    print("-" * 70)

    for model_name, features in model_features.items():

        # ----------------------------------------------------
        # Prepare model-specific complete-case dataset
        # ----------------------------------------------------

        model_df = data[
            features + [outcome_col]
        ].copy()

        # Numeric conversion
        numeric_features = [
            x for x in features
            if x != "Sex"
        ]

        for col in numeric_features:
            model_df[col] = pd.to_numeric(
                model_df[col],
                errors="coerce"
            )

        model_df[outcome_col] = pd.to_numeric(
            model_df[outcome_col],
            errors="coerce"
        )

        # Complete-case analysis for this model
        model_df = model_df.dropna(
            subset=features + [outcome_col]
        ).copy()

        model_df[outcome_col] = (
            model_df[outcome_col]
            .astype(int)
        )

        X = model_df[features].copy()
        y = model_df[outcome_col].copy()

        n_total = len(model_df)
        n_events = int(y.sum())

        print(
            f"\n{model_name}"
            f"\n  N      : {n_total}"
            f"\n  Events : {n_events}"
        )


        # ----------------------------------------------------
        # PREPROCESSING
        # ----------------------------------------------------

        numeric_cols = [
            col for col in features
            if col != "Sex"
        ]

        categorical_cols = [
            col for col in features
            if col == "Sex"
        ]

        transformers = []

        if len(numeric_cols) > 0:

            transformers.append(
                (
                    "numeric",
                    StandardScaler(),
                    numeric_cols
                )
            )

        if len(categorical_cols) > 0:

            transformers.append(
                (
                    "categorical",
                    OneHotEncoder(
                        drop="first",
                        handle_unknown="ignore"
                    ),
                    categorical_cols
                )
            )

        preprocessor = ColumnTransformer(
            transformers=transformers,
            remainder="drop"
        )


        # ----------------------------------------------------
        # LOGISTIC REGRESSION
        #
        # sklearn >=1.8:
        # penalty="l2" is deprecated.
        #
        # l1_ratio=0.0 gives equivalent L2 regularization.
        # ----------------------------------------------------

        logistic_model = LogisticRegression(
            C=1.0,
            l1_ratio=0.0,
            solver="liblinear",
            max_iter=5000,
            random_state=seed
        )


        pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("model", logistic_model)
            ]
        )


        # ----------------------------------------------------
        # REPEATED STRATIFIED CROSS-VALIDATION
        # 5 folds x 20 repeats = 100 test splits/model
        # ----------------------------------------------------

        cv = RepeatedStratifiedKFold(
            n_splits=5,
            n_repeats=20,
            random_state=seed
        )

        split_number = 0

        for train_idx, test_idx in cv.split(X, y):

            split_number += 1

            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]

            y_train = y.iloc[train_idx]
            y_test = y.iloc[test_idx]


            # ------------------------------------------------
            # Fit only on training fold
            # ------------------------------------------------

            pipeline.fit(
                X_train,
                y_train
            )


            # ------------------------------------------------
            # Test-fold predicted probabilities
            # ------------------------------------------------

            y_prob = pipeline.predict_proba(
                X_test
            )[:, 1]


            # ------------------------------------------------
            # PERFORMANCE METRICS
            # ------------------------------------------------

            roc_auc = roc_auc_score(
                y_test,
                y_prob
            )

            pr_auc = average_precision_score(
                y_test,
                y_prob
            )

            brier = brier_score_loss(
                y_test,
                y_prob
            )


            results.append(
                {
                    "Outcome": outcome_label,
                    "Model": model_name,
                    "Split": split_number,
                    "Test_N": len(y_test),
                    "Test_Events": int(y_test.sum()),
                    "ROC_AUC": roc_auc,
                    "PR_AUC": pr_auc,
                    "Brier_Score": brier
                }
            )


    return pd.DataFrame(results)


# ============================================================
# FUTURE SEVERE DENGUE
# ============================================================

cv_severe = evaluate_models_cv(
    data=severe_d34,
    outcome_col="Severe_future_outcome",
    outcome_label="Future severe dengue",
    model_features=MODEL_FEATURES,
    seed=SEED
)


# ============================================================
# FUTURE PLASMA LEAKAGE
# ============================================================

cv_leakage = evaluate_models_cv(
    data=leakage_d34,
    outcome_col="Leakage_future_outcome",
    outcome_label="Future plasma leakage",
    model_features=MODEL_FEATURES,
    seed=SEED
)


# ============================================================
# COMBINE ALL SPLIT-LEVEL RESULTS
# ============================================================

cv_results = pd.concat(
    [
        cv_severe,
        cv_leakage
    ],
    ignore_index=True
)


# ============================================================
# SUMMARY TABLE
# ============================================================

cv_summary = (
    cv_results
    .groupby(
        [
            "Outcome",
            "Model"
        ],
        as_index=False
    )
    .agg(

        ROC_AUC_Mean=(
            "ROC_AUC",
            "mean"
        ),

        ROC_AUC_SD=(
            "ROC_AUC",
            "std"
        ),

        PR_AUC_Mean=(
            "PR_AUC",
            "mean"
        ),

        PR_AUC_SD=(
            "PR_AUC",
            "std"
        ),

        Brier_Mean=(
            "Brier_Score",
            "mean"
        ),

        Brier_SD=(
            "Brier_Score",
            "std"
        )
    )
)


# ------------------------------------------------------------
# ROUND DISPLAY VALUES
# ------------------------------------------------------------

cv_summary_display = cv_summary.copy()

metric_cols = [
    "ROC_AUC_Mean",
    "ROC_AUC_SD",
    "PR_AUC_Mean",
    "PR_AUC_SD",
    "Brier_Mean",
    "Brier_SD"
]

cv_summary_display[
    metric_cols
] = cv_summary_display[
    metric_cols
].round(4)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 70)
print("CROSS-VALIDATED MODEL PERFORMANCE")
print("=" * 70)

print(
    cv_summary_display.to_string(
        index=False
    )
)


# ============================================================
# SAVE TABLES
# ============================================================

all_splits_path = (
    TABLE_DIR
    / "Table_CV_Model_Performance_AllSplits.csv"
)

summary_path = (
    TABLE_DIR
    / "Table_CV_Static_vs_Dynamic_Model_Performance.csv"
)


cv_results.to_csv(
    all_splits_path,
    index=False
)

cv_summary.to_csv(
    summary_path,
    index=False
)


print("\nSaved:")
print(all_splits_path)

print("\nSaved:")
print(summary_path)


# ============================================================
# BASIC INTEGRITY CHECKS
# ============================================================

assert len(cv_severe) == 300, (
    "Unexpected severe-dengue CV row count."
)

assert len(cv_leakage) == 300, (
    "Unexpected plasma-leakage CV row count."
)

assert cv_results[
    [
        "ROC_AUC",
        "PR_AUC",
        "Brier_Score"
    ]
].notna().all().all(), (
    "Missing cross-validation performance values detected."
)


print("\nCross-validation integrity checks passed.")

print("\nCell 8 complete.")

In [ ]:
# ============================================================
# CELL 9
# PATIENT-LEVEL OOF BOOTSTRAP:
# INCREMENTAL VALUE OF STATIC STATE AND TEMPORAL DYNAMICS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

print("=" * 78)
print("PATIENT-LEVEL OOF BOOTSTRAP OF INCREMENTAL MODEL VALUE")
print("=" * 78)

SEED = 10010
N_BOOT = 5000
N_SPLITS = 5
N_REPEATS = 20


# ============================================================
# CHECK REQUIRED OBJECTS FROM CELL 8
# ============================================================

required_objects = [
    "MODEL_FEATURES",
    "severe_d34",
    "leakage_d34",
    "TABLE_DIR"
]

for obj in required_objects:
    assert obj in globals(), f"Required object missing: {obj}"

print("\nRequired Cell 8 objects found.")


# ============================================================
# FUNCTION:
# CREATE REPEATED-CV OOF PREDICTIONS
#
# Every patient receives exactly one out-of-fold prediction
# per repeat = 20 independent OOF predictions.
#
# Final patient probability = mean of those 20 OOF predictions.
# ============================================================

def generate_repeated_oof_predictions(
    data,
    outcome_col,
    outcome_label,
    model_features,
    seed=SEED
):

    base = data.copy().reset_index(drop=True)

    # Stable patient-row identifier within this frozen cohort
    base["_Source_Row"] = np.arange(len(base))

    all_model_predictions = []

    print("\n" + "-" * 78)
    print(outcome_label.upper())
    print("-" * 78)

    for model_name, features in model_features.items():

        print(f"\nGenerating OOF predictions: {model_name}")

        model_df = base[
            ["_Source_Row"] + features + [outcome_col]
        ].copy()

        # ----------------------------------------------------
        # Numeric conversion
        # ----------------------------------------------------

        numeric_features = [
            x for x in features
            if x != "Sex"
        ]

        for col in numeric_features:
            model_df[col] = pd.to_numeric(
                model_df[col],
                errors="coerce"
            )

        model_df[outcome_col] = pd.to_numeric(
            model_df[outcome_col],
            errors="coerce"
        )

        # ----------------------------------------------------
        # Complete cases
        # ----------------------------------------------------

        model_df = (
            model_df
            .dropna(
                subset=features + [outcome_col]
            )
            .reset_index(drop=True)
        )

        model_df[outcome_col] = (
            model_df[outcome_col]
            .astype(int)
        )

        X = model_df[features].copy()
        y = model_df[outcome_col].copy()

        n = len(model_df)

        print(
            f"  Patients : {n}\n"
            f"  Events   : {int(y.sum())}"
        )


        # ----------------------------------------------------
        # PREPROCESSING
        # ----------------------------------------------------

        numeric_cols = [
            col for col in features
            if col != "Sex"
        ]

        categorical_cols = [
            col for col in features
            if col == "Sex"
        ]

        transformers = []

        if len(numeric_cols) > 0:

            transformers.append(
                (
                    "numeric",
                    StandardScaler(),
                    numeric_cols
                )
            )

        if len(categorical_cols) > 0:

            transformers.append(
                (
                    "categorical",
                    OneHotEncoder(
                        drop="first",
                        handle_unknown="ignore"
                    ),
                    categorical_cols
                )
            )

        preprocessor = ColumnTransformer(
            transformers=transformers,
            remainder="drop"
        )


        # ----------------------------------------------------
        # MODEL
        # sklearn >= 1.8 compatible L2 logistic regression
        # ----------------------------------------------------

        model = LogisticRegression(
            C=1.0,
            l1_ratio=0.0,
            solver="liblinear",
            max_iter=5000,
            random_state=seed
        )

        pipe = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("model", model)
            ]
        )


        # ----------------------------------------------------
        # REPEATED STRATIFIED CV
        # ----------------------------------------------------

        cv = RepeatedStratifiedKFold(
            n_splits=N_SPLITS,
            n_repeats=N_REPEATS,
            random_state=seed
        )

        pred_sum = np.zeros(n, dtype=float)
        pred_count = np.zeros(n, dtype=int)


        for train_idx, test_idx in cv.split(X, y):

            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]

            y_train = y.iloc[train_idx]

            pipe.fit(
                X_train,
                y_train
            )

            test_prob = pipe.predict_proba(
                X_test
            )[:, 1]

            pred_sum[test_idx] += test_prob
            pred_count[test_idx] += 1


        # ----------------------------------------------------
        # INTEGRITY:
        # EACH PATIENT SHOULD HAVE 20 OOF PREDICTIONS
        # ----------------------------------------------------

        assert np.all(
            pred_count == N_REPEATS
        ), (
            f"{model_name}: each patient did not receive "
            f"exactly {N_REPEATS} OOF predictions."
        )

        mean_oof_probability = (
            pred_sum / pred_count
        )


        model_predictions = pd.DataFrame(
            {
                "_Source_Row":
                    model_df["_Source_Row"].values,

                "Outcome":
                    outcome_label,

                "Outcome_Value":
                    y.values,

                "Model":
                    model_name,

                "OOF_Probability":
                    mean_oof_probability,

                "OOF_Prediction_Count":
                    pred_count
            }
        )

        all_model_predictions.append(
            model_predictions
        )


    return pd.concat(
        all_model_predictions,
        ignore_index=True
    )


# ============================================================
# GENERATE OOF PREDICTIONS
# ============================================================

oof_severe = generate_repeated_oof_predictions(
    data=severe_d34,
    outcome_col="Severe_future_outcome",
    outcome_label="Future severe dengue",
    model_features=MODEL_FEATURES,
    seed=SEED
)

oof_leakage = generate_repeated_oof_predictions(
    data=leakage_d34,
    outcome_col="Leakage_future_outcome",
    outcome_label="Future plasma leakage",
    model_features=MODEL_FEATURES,
    seed=SEED
)


oof_all = pd.concat(
    [
        oof_severe,
        oof_leakage
    ],
    ignore_index=True
)


# ============================================================
# PERFORMANCE FROM PATIENT-LEVEL MEAN OOF PROBABILITIES
# ============================================================

def summarize_oof_performance(oof_df):

    rows = []

    for (outcome, model_name), group in (
        oof_df.groupby(
            ["Outcome", "Model"]
        )
    ):

        y = group[
            "Outcome_Value"
        ].to_numpy()

        p = group[
            "OOF_Probability"
        ].to_numpy()

        rows.append(
            {
                "Outcome": outcome,
                "Model": model_name,
                "N": len(group),
                "Events": int(y.sum()),

                "ROC_AUC":
                    roc_auc_score(y, p),

                "PR_AUC":
                    average_precision_score(y, p),

                "Brier_Score":
                    brier_score_loss(y, p)
            }
        )

    return pd.DataFrame(rows)


oof_performance = summarize_oof_performance(
    oof_all
)


print("\n" + "=" * 78)
print("PATIENT-LEVEL MEAN OOF PERFORMANCE")
print("=" * 78)

display_perf = oof_performance.copy()

for c in [
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score"
]:
    display_perf[c] = (
        display_perf[c]
        .round(4)
    )

print(
    display_perf.to_string(
        index=False
    )
)


# ============================================================
# CONVERT TO WIDE FORMAT FOR PAIRED COMPARISONS
# ============================================================

def make_prediction_wide(oof_df, outcome_name):

    x = oof_df[
        oof_df["Outcome"] == outcome_name
    ].copy()

    wide = (
        x.pivot(
            index=[
                "_Source_Row",
                "Outcome_Value"
            ],
            columns="Model",
            values="OOF_Probability"
        )
        .reset_index()
    )

    required_models = [
        "A_Demographics",
        "B_Static_DOI4",
        "C_Static_plus_Dynamics"
    ]

    for model in required_models:
        assert model in wide.columns, (
            f"Missing model predictions: {model}"
        )

    wide = wide.dropna(
        subset=required_models
    ).reset_index(drop=True)

    return wide


wide_severe = make_prediction_wide(
    oof_all,
    "Future severe dengue"
)

wide_leakage = make_prediction_wide(
    oof_all,
    "Future plasma leakage"
)


# ============================================================
# STRATIFIED PATIENT-LEVEL BOOTSTRAP
#
# Positive delta ROC/PR = new model better
# Positive Brier improvement = new model better
# ============================================================

def paired_bootstrap_comparison(
    wide,
    outcome_label,
    reference_model,
    new_model,
    n_boot=N_BOOT,
    seed=SEED
):

    y = wide[
        "Outcome_Value"
    ].to_numpy().astype(int)

    p_ref = wide[
        reference_model
    ].to_numpy()

    p_new = wide[
        new_model
    ].to_numpy()

    positive_idx = np.where(
        y == 1
    )[0]

    negative_idx = np.where(
        y == 0
    )[0]

    rng = np.random.default_rng(seed)

    # --------------------------------------------------------
    # OBSERVED DIFFERENCES
    # --------------------------------------------------------

    observed_roc_ref = roc_auc_score(
        y,
        p_ref
    )

    observed_roc_new = roc_auc_score(
        y,
        p_new
    )

    observed_pr_ref = average_precision_score(
        y,
        p_ref
    )

    observed_pr_new = average_precision_score(
        y,
        p_new
    )

    observed_brier_ref = brier_score_loss(
        y,
        p_ref
    )

    observed_brier_new = brier_score_loss(
        y,
        p_new
    )


    observed_delta_roc = (
        observed_roc_new
        - observed_roc_ref
    )

    observed_delta_pr = (
        observed_pr_new
        - observed_pr_ref
    )

    # Positive = improved calibration/error performance
    observed_brier_improvement = (
        observed_brier_ref
        - observed_brier_new
    )


    # --------------------------------------------------------
    # BOOTSTRAP DISTRIBUTIONS
    # --------------------------------------------------------

    boot_delta_roc = np.empty(
        n_boot
    )

    boot_delta_pr = np.empty(
        n_boot
    )

    boot_brier_improvement = np.empty(
        n_boot
    )


    for b in range(n_boot):

        sampled_positive = rng.choice(
            positive_idx,
            size=len(positive_idx),
            replace=True
        )

        sampled_negative = rng.choice(
            negative_idx,
            size=len(negative_idx),
            replace=True
        )

        sampled_idx = np.concatenate(
            [
                sampled_positive,
                sampled_negative
            ]
        )

        y_b = y[
            sampled_idx
        ]

        ref_b = p_ref[
            sampled_idx
        ]

        new_b = p_new[
            sampled_idx
        ]


        boot_delta_roc[b] = (
            roc_auc_score(
                y_b,
                new_b
            )
            -
            roc_auc_score(
                y_b,
                ref_b
            )
        )


        boot_delta_pr[b] = (
            average_precision_score(
                y_b,
                new_b
            )
            -
            average_precision_score(
                y_b,
                ref_b
            )
        )


        boot_brier_improvement[b] = (
            brier_score_loss(
                y_b,
                ref_b
            )
            -
            brier_score_loss(
                y_b,
                new_b
            )
        )


    # --------------------------------------------------------
    # 95% PERCENTILE CONFIDENCE INTERVALS
    # --------------------------------------------------------

    roc_low, roc_high = np.percentile(
        boot_delta_roc,
        [2.5, 97.5]
    )

    pr_low, pr_high = np.percentile(
        boot_delta_pr,
        [2.5, 97.5]
    )

    brier_low, brier_high = np.percentile(
        boot_brier_improvement,
        [2.5, 97.5]
    )


    return {
        "Outcome":
            outcome_label,

        "Reference_Model":
            reference_model,

        "New_Model":
            new_model,

        "N":
            len(y),

        "Events":
            int(y.sum()),

        "Delta_ROC_AUC":
            observed_delta_roc,

        "Delta_ROC_AUC_CI_Low":
            roc_low,

        "Delta_ROC_AUC_CI_High":
            roc_high,

        "Delta_PR_AUC":
            observed_delta_pr,

        "Delta_PR_AUC_CI_Low":
            pr_low,

        "Delta_PR_AUC_CI_High":
            pr_high,

        "Brier_Improvement":
            observed_brier_improvement,

        "Brier_Improvement_CI_Low":
            brier_low,

        "Brier_Improvement_CI_High":
            brier_high,

        "Bootstrap_Replicates":
            n_boot
    }


# ============================================================
# RUN THE THREE PRE-SPECIFIED MODEL COMPARISONS
# ============================================================

comparisons = [
    (
        "A_Demographics",
        "B_Static_DOI4"
    ),
    (
        "B_Static_DOI4",
        "C_Static_plus_Dynamics"
    ),
    (
        "A_Demographics",
        "C_Static_plus_Dynamics"
    )
]


bootstrap_rows = []


for outcome_label, wide_df in [

    (
        "Future severe dengue",
        wide_severe
    ),

    (
        "Future plasma leakage",
        wide_leakage
    )
]:

    print(
        "\nBootstrapping:",
        outcome_label
    )

    for reference_model, new_model in comparisons:

        print(
            "  ",
            reference_model,
            "->",
            new_model
        )

        result = paired_bootstrap_comparison(
            wide=wide_df,
            outcome_label=outcome_label,
            reference_model=reference_model,
            new_model=new_model,
            n_boot=N_BOOT,
            seed=SEED
        )

        bootstrap_rows.append(
            result
        )


bootstrap_results = pd.DataFrame(
    bootstrap_rows
)


# ============================================================
# DISPLAY
# ============================================================

display_bootstrap = (
    bootstrap_results.copy()
)

numeric_cols = [
    "Delta_ROC_AUC",
    "Delta_ROC_AUC_CI_Low",
    "Delta_ROC_AUC_CI_High",
    "Delta_PR_AUC",
    "Delta_PR_AUC_CI_Low",
    "Delta_PR_AUC_CI_High",
    "Brier_Improvement",
    "Brier_Improvement_CI_Low",
    "Brier_Improvement_CI_High"
]

display_bootstrap[
    numeric_cols
] = (
    display_bootstrap[
        numeric_cols
    ]
    .round(4)
)


print("\n" + "=" * 78)
print("PAIRED PATIENT-LEVEL BOOTSTRAP RESULTS")
print("=" * 78)

print(
    display_bootstrap.to_string(
        index=False
    )
)


# ============================================================
# KEY C vs B COMPARISON ONLY
# ============================================================

key_incremental = (
    display_bootstrap[
        (
            display_bootstrap[
                "Reference_Model"
            ]
            == "B_Static_DOI4"
        )
        &
        (
            display_bootstrap[
                "New_Model"
            ]
            == "C_Static_plus_Dynamics"
        )
    ]
    .copy()
)


print("\n" + "=" * 78)
print("KEY TEST: DOES TEMPORAL DYNAMICS ADD VALUE BEYOND DOI-4 STATE?")
print("=" * 78)

print(
    key_incremental.to_string(
        index=False
    )
)


# ============================================================
# SAVE OUTPUTS
# ============================================================

oof_path = (
    TABLE_DIR
    / "Table_RepeatedCV_PatientLevel_OOF_Predictions.csv"
)

oof_performance_path = (
    TABLE_DIR
    / "Table_RepeatedCV_PatientLevel_OOF_Performance.csv"
)

bootstrap_path = (
    TABLE_DIR
    / "Table_Paired_Bootstrap_Incremental_Model_Value.csv"
)


oof_all.to_csv(
    oof_path,
    index=False
)

oof_performance.to_csv(
    oof_performance_path,
    index=False
)

bootstrap_results.to_csv(
    bootstrap_path,
    index=False
)


print("\nSaved:")
print(oof_path)

print("\nSaved:")
print(oof_performance_path)

print("\nSaved:")
print(bootstrap_path)


# ============================================================
# FINAL INTEGRITY CHECKS
# ============================================================

assert (
    oof_all[
        "OOF_Prediction_Count"
    ]
    == N_REPEATS
).all(), (
    "OOF prediction-count integrity failure."
)

assert (
    oof_all[
        "OOF_Probability"
    ]
    .between(0, 1)
    .all()
), (
    "Invalid OOF probabilities detected."
)

assert len(
    bootstrap_results
) == 6, (
    "Unexpected number of bootstrap comparisons."
)


print("\nAll patient-level OOF and bootstrap integrity checks passed.")

print("\nCell 9 complete.")

In [ ]:
# ============================================================
# CELL 10
# CALIBRATION ANALYSIS + PUBLICATION-QUALITY MODEL COMPARISON
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

print("=" * 82)
print("CALIBRATION AND FINAL MODEL COMPARISON")
print("=" * 82)


# ============================================================
# SETTINGS
# ============================================================

SEED = 10010

FIGURE_DIR = (
    TABLE_DIR.parent
    / "figures"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PRIMARY_MODEL = "B_Static_DOI4"

MODEL_ORDER = [
    "A_Demographics",
    "B_Static_DOI4",
    "C_Static_plus_Dynamics"
]

MODEL_LABELS = {
    "A_Demographics":
        "A: Demographics",

    "B_Static_DOI4":
        "B: Static DOI-4 state",

    "C_Static_plus_Dynamics":
        "C: Static + DOI 3→4 dynamics"
}


# ============================================================
# CALIBRATION INTERCEPT AND SLOPE
#
# Calibration intercept:
#     ideal = 0
#
# Calibration slope:
#     ideal = 1
#
# Uses patient-level mean OOF probabilities from Cell 9.
# ============================================================

def calibration_intercept_slope(
    y,
    p
):

    y = np.asarray(
        y,
        dtype=int
    )

    p = np.asarray(
        p,
        dtype=float
    )

    # Avoid infinite logits
    eps = 1e-6

    p = np.clip(
        p,
        eps,
        1 - eps
    )

    logit_p = np.log(
        p / (1 - p)
    )


    # --------------------------------------------------------
    # Calibration intercept
    #
    # Outcome ~ offset(logit prediction)
    #
    # Numerically estimate intercept using logistic regression
    # with fixed coefficient approximately represented by
    # recalibrating predicted odds.
    # --------------------------------------------------------

    def intercept_score(a):

        calibrated_p = 1 / (
            1
            + np.exp(
                -(a + logit_p)
            )
        )

        return np.sum(
            y - calibrated_p
        )


    # Bisection solution
    low = -10.0
    high = 10.0

    for _ in range(100):

        mid = (
            low + high
        ) / 2

        if intercept_score(mid) > 0:
            low = mid
        else:
            high = mid

    cal_intercept = (
        low + high
    ) / 2


    # --------------------------------------------------------
    # Calibration slope
    #
    # Logistic regression:
    # outcome ~ logit(predicted probability)
    # --------------------------------------------------------

    slope_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=5000,
        random_state=SEED
    )

    slope_model.fit(
        logit_p.reshape(-1, 1),
        y
    )

    cal_slope = float(
        slope_model.coef_[0][0]
    )


    return (
        cal_intercept,
        cal_slope
    )


# ============================================================
# CALIBRATION TABLE
# ============================================================

calibration_rows = []


for (
    outcome_name,
    model_name
), group in oof_all.groupby(
    [
        "Outcome",
        "Model"
    ]
):

    y = (
        group[
            "Outcome_Value"
        ]
        .to_numpy()
        .astype(int)
    )

    p = (
        group[
            "OOF_Probability"
        ]
        .to_numpy()
        .astype(float)
    )


    cal_intercept, cal_slope = (
        calibration_intercept_slope(
            y,
            p
        )
    )


    calibration_rows.append(
        {
            "Outcome":
                outcome_name,

            "Model":
                model_name,

            "N":
                len(y),

            "Events":
                int(y.sum()),

            "Event_Rate":
                y.mean(),

            "Mean_Predicted_Risk":
                p.mean(),

            "ROC_AUC":
                roc_auc_score(
                    y,
                    p
                ),

            "PR_AUC":
                average_precision_score(
                    y,
                    p
                ),

            "Brier_Score":
                brier_score_loss(
                    y,
                    p
                ),

            "Calibration_Intercept":
                cal_intercept,

            "Calibration_Slope":
                cal_slope
        }
    )


calibration_table = pd.DataFrame(
    calibration_rows
)


# ============================================================
# DISPLAY
# ============================================================

display_calibration = (
    calibration_table.copy()
)

display_cols = [
    "Event_Rate",
    "Mean_Predicted_Risk",
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score",
    "Calibration_Intercept",
    "Calibration_Slope"
]

display_calibration[
    display_cols
] = (
    display_calibration[
        display_cols
    ]
    .round(4)
)


print("\n" + "=" * 82)
print("OOF DISCRIMINATION AND CALIBRATION")
print("=" * 82)

print(
    display_calibration.to_string(
        index=False
    )
)


# ============================================================
# CALIBRATION-BIN FUNCTION
# ============================================================

def calibration_bins(
    group,
    n_bins
):

    x = group[
        [
            "Outcome_Value",
            "OOF_Probability"
        ]
    ].copy()

    # Quantile bins
    x[
        "Risk_Bin"
    ] = pd.qcut(
        x[
            "OOF_Probability"
        ],
        q=n_bins,
        duplicates="drop"
    )


    summary = (
        x.groupby(
            "Risk_Bin",
            observed=True
        )
        .agg(
            Mean_Predicted_Risk=(
                "OOF_Probability",
                "mean"
            ),
            Observed_Event_Rate=(
                "Outcome_Value",
                "mean"
            ),
            N=(
                "Outcome_Value",
                "size"
            ),
            Events=(
                "Outcome_Value",
                "sum"
            )
        )
        .reset_index()
    )


    return summary


# ============================================================
# CREATE CALIBRATION BIN TABLES
#
# Severe dengue:
# fewer bins because only 37 events
#
# Plasma leakage:
# deciles are acceptable
# ============================================================

calibration_bin_tables = []


for outcome_name in [
    "Future severe dengue",
    "Future plasma leakage"
]:

    if outcome_name == "Future severe dengue":
        n_bins = 5
    else:
        n_bins = 10


    for model_name in MODEL_ORDER:

        group = oof_all[
            (
                oof_all[
                    "Outcome"
                ]
                == outcome_name
            )
            &
            (
                oof_all[
                    "Model"
                ]
                == model_name
            )
        ].copy()


        bin_table = calibration_bins(
            group,
            n_bins=n_bins
        )


        bin_table[
            "Outcome"
        ] = outcome_name

        bin_table[
            "Model"
        ] = model_name


        calibration_bin_tables.append(
            bin_table
        )


calibration_bins_all = pd.concat(
    calibration_bin_tables,
    ignore_index=True
)


# ============================================================
# FINAL PRIMARY-MODEL CALIBRATION PLOTS
# ============================================================

for outcome_name in [
    "Future severe dengue",
    "Future plasma leakage"
]:

    primary_group = oof_all[
        (
            oof_all[
                "Outcome"
            ]
            == outcome_name
        )
        &
        (
            oof_all[
                "Model"
            ]
            == PRIMARY_MODEL
        )
    ].copy()


    if outcome_name == "Future severe dengue":
        n_bins = 5
    else:
        n_bins = 10


    bin_table = calibration_bins(
        primary_group,
        n_bins=n_bins
    )


    fig, ax = plt.subplots(
        figsize=(7.5, 7.0)
    )


    # Perfect calibration reference
    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
        label="Perfect calibration"
    )


    # Observed calibration
    ax.plot(
        bin_table[
            "Mean_Predicted_Risk"
        ],
        bin_table[
            "Observed_Event_Rate"
        ],
        marker="o",
        linewidth=2,
        markersize=7,
        label="Observed calibration"
    )


    ax.set_xlabel(
        "Mean predicted probability",
        fontsize=12
    )

    ax.set_ylabel(
        "Observed event probability",
        fontsize=12
    )

    ax.set_title(
        (
            f"{outcome_name}\n"
            "Primary Model B: static DOI-4 host–virus state"
        ),
        fontsize=13
    )


    max_value = max(
        0.05,
        float(
            max(
                bin_table[
                    "Mean_Predicted_Risk"
                ].max(),

                bin_table[
                    "Observed_Event_Rate"
                ].max()
            )
        ) * 1.15
    )


    ax.set_xlim(
        0,
        max_value
    )

    ax.set_ylim(
        0,
        max_value
    )


    ax.legend(
        frameon=False
    )

    ax.grid(
        alpha=0.2
    )


    plt.tight_layout()


    safe_name = (
        outcome_name
        .lower()
        .replace(" ", "_")
    )


    png_path = (
        FIGURE_DIR
        / f"Figure_Calibration_{safe_name}.png"
    )

    svg_path = (
        FIGURE_DIR
        / f"Figure_Calibration_{safe_name}.svg"
    )


    plt.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight"
    )

    plt.savefig(
        svg_path,
        bbox_inches="tight"
    )

    plt.show()


    print("\nSaved:")
    print(png_path)

    print("Saved:")
    print(svg_path)


# ============================================================
# MODEL COMPARISON FIGURE
#
# Patient-level mean OOF metrics
# ============================================================

plot_data = (
    oof_performance.copy()
)

plot_data[
    "Model_Label"
] = (
    plot_data[
        "Model"
    ]
    .map(
        MODEL_LABELS
    )
)


for metric, ylabel, filename in [

    (
        "ROC_AUC",
        "ROC-AUC",
        "Figure_Model_Comparison_ROC_AUC"
    ),

    (
        "PR_AUC",
        "Precision–Recall AUC",
        "Figure_Model_Comparison_PR_AUC"
    ),

    (
        "Brier_Score",
        "Brier score",
        "Figure_Model_Comparison_Brier"
    )
]:

    fig, ax = plt.subplots(
        figsize=(9.0, 6.0)
    )


    outcomes = [
        "Future severe dengue",
        "Future plasma leakage"
    ]


    x = np.arange(
        len(MODEL_ORDER)
    )

    width = 0.34


    for i, outcome_name in enumerate(
        outcomes
    ):

        subset = (
            plot_data[
                plot_data[
                    "Outcome"
                ]
                == outcome_name
            ]
            .set_index(
                "Model"
            )
            .reindex(
                MODEL_ORDER
            )
        )


        offset = (
            -width / 2
            if i == 0
            else width / 2
        )


        ax.bar(
            x + offset,
            subset[
                metric
            ].values,
            width=width,
            label=outcome_name
        )


    ax.set_xticks(
        x
    )

    ax.set_xticklabels(
        [
            MODEL_LABELS[m]
            for m in MODEL_ORDER
        ],
        rotation=10,
        ha="right"
    )


    ax.set_ylabel(
        ylabel,
        fontsize=12
    )

    ax.set_title(
        "Out-of-fold predictive performance",
        fontsize=13
    )


    ax.legend(
        frameon=False
    )

    ax.grid(
        axis="y",
        alpha=0.2
    )


    plt.tight_layout()


    png_path = (
        FIGURE_DIR
        / f"{filename}.png"
    )

    svg_path = (
        FIGURE_DIR
        / f"{filename}.svg"
    )


    plt.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight"
    )

    plt.savefig(
        svg_path,
        bbox_inches="tight"
    )

    plt.show()


    print("\nSaved:")
    print(png_path)

    print("Saved:")
    print(svg_path)


# ============================================================
# SAVE CALIBRATION TABLES
# ============================================================

calibration_path = (
    TABLE_DIR
    / "Table_OOF_Calibration_and_Performance.csv"
)

calibration_bins_path = (
    TABLE_DIR
    / "Table_OOF_Calibration_Bins.csv"
)


calibration_table.to_csv(
    calibration_path,
    index=False
)

calibration_bins_all.to_csv(
    calibration_bins_path,
    index=False
)


print("\nSaved:")
print(calibration_path)

print("\nSaved:")
print(calibration_bins_path)


# ============================================================
# PRIMARY MODEL SUMMARY
# ============================================================

primary_summary = (
    display_calibration[
        display_calibration[
            "Model"
        ]
        == PRIMARY_MODEL
    ]
    .copy()
)


print("\n" + "=" * 82)
print("PRIMARY MODEL B — FINAL OOF SUMMARY")
print("=" * 82)

print(
    primary_summary.to_string(
        index=False
    )
)


# ============================================================
# INTEGRITY CHECKS
# ============================================================

assert (
    calibration_table[
        "Mean_Predicted_Risk"
    ]
    .between(
        0,
        1
    )
    .all()
)

assert (
    calibration_table[
        "ROC_AUC"
    ]
    .between(
        0,
        1
    )
    .all()
)

assert (
    calibration_table[
        "PR_AUC"
    ]
    .between(
        0,
        1
    )
    .all()
)

assert (
    calibration_table[
        "Brier_Score"
    ]
    .between(
        0,
        1
    )
    .all()
)


print(
    "\nCalibration and figure integrity checks passed."
)

print("\nCell 10 complete.")

In [ ]:
# ============================================================
# CELL 11
# PRIMARY MODEL B:
# BOOTSTRAP UNCERTAINTY + RISK CONCENTRATION ANALYSIS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

print("=" * 84)
print("PRIMARY MODEL B — BOOTSTRAP UNCERTAINTY AND RISK CONCENTRATION")
print("=" * 84)


# ============================================================
# SETTINGS
# ============================================================

SEED = 10010
N_BOOT = 5000

PRIMARY_MODEL = "B_Static_DOI4"

OUTCOME_ORDER = [
    "Future severe dengue",
    "Future plasma leakage"
]


# ============================================================
# CHECK REQUIRED OBJECTS
# ============================================================

required_objects = [
    "oof_all",
    "TABLE_DIR",
    "FIGURE_DIR"
]

for obj in required_objects:
    assert obj in globals(), (
        f"Required object missing: {obj}"
    )

print("\nRequired objects found.")


# ============================================================
# CALIBRATION METRICS
# ============================================================

def calibration_metrics(y, p):

    y = np.asarray(
        y,
        dtype=int
    )

    p = np.asarray(
        p,
        dtype=float
    )

    eps = 1e-6

    p = np.clip(
        p,
        eps,
        1 - eps
    )

    logit_p = np.log(
        p / (1 - p)
    )


    # --------------------------------------------------------
    # CALIBRATION-IN-THE-LARGE
    # slope fixed at 1
    # --------------------------------------------------------

    def score(a):

        z = a + logit_p

        # numerically stable logistic transform
        calibrated = np.where(
            z >= 0,
            1 / (1 + np.exp(-z)),
            np.exp(z) / (1 + np.exp(z))
        )

        return np.sum(
            y - calibrated
        )


    low = -15.0
    high = 15.0

    for _ in range(100):

        mid = (
            low + high
        ) / 2

        if score(mid) > 0:
            low = mid
        else:
            high = mid

    cal_intercept = (
        low + high
    ) / 2


    # --------------------------------------------------------
    # CALIBRATION SLOPE
    # logistic outcome ~ logit(prediction)
    # --------------------------------------------------------

    cal_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=5000,
        random_state=SEED
    )

    cal_model.fit(
        logit_p.reshape(-1, 1),
        y
    )

    cal_slope = float(
        cal_model.coef_[0][0]
    )


    return (
        cal_intercept,
        cal_slope
    )


# ============================================================
# STRATIFIED BOOTSTRAP
# ============================================================

def bootstrap_primary_model(
    group,
    outcome_name,
    n_boot=N_BOOT,
    seed=SEED
):

    y = (
        group[
            "Outcome_Value"
        ]
        .to_numpy()
        .astype(int)
    )

    p = (
        group[
            "OOF_Probability"
        ]
        .to_numpy()
        .astype(float)
    )


    positive_idx = np.where(
        y == 1
    )[0]

    negative_idx = np.where(
        y == 0
    )[0]


    rng = np.random.default_rng(
        seed
    )


    # --------------------------------------------------------
    # OBSERVED VALUES
    # --------------------------------------------------------

    roc_obs = roc_auc_score(
        y,
        p
    )

    pr_obs = average_precision_score(
        y,
        p
    )

    brier_obs = brier_score_loss(
        y,
        p
    )

    intercept_obs, slope_obs = (
        calibration_metrics(
            y,
            p
        )
    )


    # --------------------------------------------------------
    # BOOTSTRAP ARRAYS
    # --------------------------------------------------------

    roc_boot = np.empty(
        n_boot
    )

    pr_boot = np.empty(
        n_boot
    )

    brier_boot = np.empty(
        n_boot
    )

    intercept_boot = np.empty(
        n_boot
    )

    slope_boot = np.empty(
        n_boot
    )


    # --------------------------------------------------------
    # BOOTSTRAP LOOP
    # --------------------------------------------------------

    for b in range(n_boot):

        sampled_positive = rng.choice(
            positive_idx,
            size=len(positive_idx),
            replace=True
        )

        sampled_negative = rng.choice(
            negative_idx,
            size=len(negative_idx),
            replace=True
        )

        sampled_idx = np.concatenate(
            [
                sampled_positive,
                sampled_negative
            ]
        )


        y_b = y[
            sampled_idx
        ]

        p_b = p[
            sampled_idx
        ]


        roc_boot[b] = roc_auc_score(
            y_b,
            p_b
        )

        pr_boot[b] = average_precision_score(
            y_b,
            p_b
        )

        brier_boot[b] = brier_score_loss(
            y_b,
            p_b
        )


        intercept_b, slope_b = (
            calibration_metrics(
                y_b,
                p_b
            )
        )

        intercept_boot[b] = intercept_b
        slope_boot[b] = slope_b


    # --------------------------------------------------------
    # 95% PERCENTILE CIs
    # --------------------------------------------------------

    def ci(x):

        return np.percentile(
            x,
            [2.5, 97.5]
        )


    roc_low, roc_high = ci(
        roc_boot
    )

    pr_low, pr_high = ci(
        pr_boot
    )

    brier_low, brier_high = ci(
        brier_boot
    )

    intercept_low, intercept_high = ci(
        intercept_boot
    )

    slope_low, slope_high = ci(
        slope_boot
    )


    return {
        "Outcome":
            outcome_name,

        "N":
            len(y),

        "Events":
            int(y.sum()),

        "Event_Rate":
            y.mean(),

        "Mean_Predicted_Risk":
            p.mean(),

        "ROC_AUC":
            roc_obs,

        "ROC_AUC_CI_Low":
            roc_low,

        "ROC_AUC_CI_High":
            roc_high,

        "PR_AUC":
            pr_obs,

        "PR_AUC_CI_Low":
            pr_low,

        "PR_AUC_CI_High":
            pr_high,

        "Brier_Score":
            brier_obs,

        "Brier_CI_Low":
            brier_low,

        "Brier_CI_High":
            brier_high,

        "Calibration_Intercept":
            intercept_obs,

        "Calibration_Intercept_CI_Low":
            intercept_low,

        "Calibration_Intercept_CI_High":
            intercept_high,

        "Calibration_Slope":
            slope_obs,

        "Calibration_Slope_CI_Low":
            slope_low,

        "Calibration_Slope_CI_High":
            slope_high,

        "Bootstrap_Replicates":
            n_boot
    }


# ============================================================
# RUN BOOTSTRAP FOR BOTH OUTCOMES
# ============================================================

bootstrap_rows = []


for outcome_name in OUTCOME_ORDER:

    print(
        "\nBootstrapping:",
        outcome_name
    )

    group = oof_all[
        (
            oof_all[
                "Outcome"
            ]
            == outcome_name
        )
        &
        (
            oof_all[
                "Model"
            ]
            == PRIMARY_MODEL
        )
    ].copy()


    result = bootstrap_primary_model(
        group=group,
        outcome_name=outcome_name,
        n_boot=N_BOOT,
        seed=SEED
    )

    bootstrap_rows.append(
        result
    )


primary_bootstrap = pd.DataFrame(
    bootstrap_rows
)


# ============================================================
# DISPLAY BOOTSTRAP RESULTS
# ============================================================

display_bootstrap = (
    primary_bootstrap.copy()
)

numeric_cols = [
    "Event_Rate",
    "Mean_Predicted_Risk",

    "ROC_AUC",
    "ROC_AUC_CI_Low",
    "ROC_AUC_CI_High",

    "PR_AUC",
    "PR_AUC_CI_Low",
    "PR_AUC_CI_High",

    "Brier_Score",
    "Brier_CI_Low",
    "Brier_CI_High",

    "Calibration_Intercept",
    "Calibration_Intercept_CI_Low",
    "Calibration_Intercept_CI_High",

    "Calibration_Slope",
    "Calibration_Slope_CI_Low",
    "Calibration_Slope_CI_High"
]


display_bootstrap[
    numeric_cols
] = (
    display_bootstrap[
        numeric_cols
    ]
    .round(4)
)


print("\n" + "=" * 84)
print("PRIMARY MODEL B — BOOTSTRAP 95% CONFIDENCE INTERVALS")
print("=" * 84)

print(
    display_bootstrap.to_string(
        index=False
    )
)


# ============================================================
# RISK CONCENTRATION ANALYSIS
#
# No outcome-derived thresholds are optimized.
# Patients are simply ranked by OOF predicted risk.
#
# We evaluate top:
# 10%
# 20%
# 30%
# ============================================================

risk_concentration_rows = []


for outcome_name in OUTCOME_ORDER:

    group = (
        oof_all[
            (
                oof_all[
                    "Outcome"
                ]
                == outcome_name
            )
            &
            (
                oof_all[
                    "Model"
                ]
                == PRIMARY_MODEL
            )
        ]
        .copy()
        .sort_values(
            "OOF_Probability",
            ascending=False
        )
        .reset_index(drop=True)
    )


    total_n = len(
        group
    )

    total_events = int(
        group[
            "Outcome_Value"
        ].sum()
    )

    overall_event_rate = (
        total_events
        / total_n
    )


    for top_fraction in [
        0.10,
        0.20,
        0.30
    ]:

        top_n = int(
            np.ceil(
                total_n
                * top_fraction
            )
        )

        top = group.iloc[
            :top_n
        ].copy()

        remainder = group.iloc[
            top_n:
        ].copy()


        top_events = int(
            top[
                "Outcome_Value"
            ].sum()
        )

        remainder_events = int(
            remainder[
                "Outcome_Value"
            ].sum()
        )


        top_event_rate = (
            top_events
            / len(top)
        )

        remainder_event_rate = (
            remainder_events
            / len(remainder)
        )


        event_capture = (
            top_events
            / total_events
            if total_events > 0
            else np.nan
        )


        risk_lift = (
            top_event_rate
            / overall_event_rate
            if overall_event_rate > 0
            else np.nan
        )


        relative_risk = (
            top_event_rate
            / remainder_event_rate
            if remainder_event_rate > 0
            else np.inf
        )


        threshold_probability = float(
            top[
                "OOF_Probability"
            ].min()
        )


        risk_concentration_rows.append(
            {
                "Outcome":
                    outcome_name,

                "Top_Risk_Fraction":
                    top_fraction,

                "Top_Risk_Percent":
                    int(
                        top_fraction
                        * 100
                    ),

                "Risk_Threshold":
                    threshold_probability,

                "Top_N":
                    len(top),

                "Top_Events":
                    top_events,

                "Top_Event_Rate":
                    top_event_rate,

                "Remaining_N":
                    len(remainder),

                "Remaining_Events":
                    remainder_events,

                "Remaining_Event_Rate":
                    remainder_event_rate,

                "Event_Capture":
                    event_capture,

                "Risk_Lift":
                    risk_lift,

                "Relative_Risk_vs_Remaining":
                    relative_risk
            }
        )


risk_concentration = pd.DataFrame(
    risk_concentration_rows
)


display_risk = (
    risk_concentration.copy()
)

risk_numeric_cols = [
    "Risk_Threshold",
    "Top_Event_Rate",
    "Remaining_Event_Rate",
    "Event_Capture",
    "Risk_Lift",
    "Relative_Risk_vs_Remaining"
]

display_risk[
    risk_numeric_cols
] = (
    display_risk[
        risk_numeric_cols
    ]
    .round(4)
)


print("\n" + "=" * 84)
print("PRIMARY MODEL B — RISK CONCENTRATION")
print("=" * 84)

print(
    display_risk.to_string(
        index=False
    )
)


# ============================================================
# CUMULATIVE EVENT-CAPTURE CURVES
# ============================================================

capture_curve_rows = []


for outcome_name in OUTCOME_ORDER:

    group = (
        oof_all[
            (
                oof_all[
                    "Outcome"
                ]
                == outcome_name
            )
            &
            (
                oof_all[
                    "Model"
                ]
                == PRIMARY_MODEL
            )
        ]
        .copy()
        .sort_values(
            "OOF_Probability",
            ascending=False
        )
        .reset_index(drop=True)
    )


    total_events = (
        group[
            "Outcome_Value"
        ].sum()
    )


    group[
        "Population_Fraction"
    ] = (
        np.arange(
            1,
            len(group) + 1
        )
        / len(group)
    )


    group[
        "Cumulative_Event_Capture"
    ] = (
        group[
            "Outcome_Value"
        ]
        .cumsum()
        / total_events
    )


    temp = group[
        [
            "Population_Fraction",
            "Cumulative_Event_Capture"
        ]
    ].copy()

    temp[
        "Outcome"
    ] = outcome_name


    capture_curve_rows.append(
        temp
    )


capture_curves = pd.concat(
    capture_curve_rows,
    ignore_index=True
)


# ============================================================
# PLOT EACH OUTCOME SEPARATELY
# ============================================================

for outcome_name in OUTCOME_ORDER:

    temp = capture_curves[
        capture_curves[
            "Outcome"
        ]
        == outcome_name
    ].copy()


    fig, ax = plt.subplots(
        figsize=(7.5, 6.5)
    )


    # Random-ranking reference
    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
        label="Random ranking"
    )


    ax.plot(
        temp[
            "Population_Fraction"
        ],
        temp[
            "Cumulative_Event_Capture"
        ],
        linewidth=2,
        label="Primary Model B"
    )


    ax.set_xlabel(
        "Fraction of patients ranked highest risk",
        fontsize=12
    )

    ax.set_ylabel(
        "Fraction of future events captured",
        fontsize=12
    )

    ax.set_title(
        (
            f"Risk concentration — {outcome_name}\n"
            "Primary DOI-4 landmark model"
        ),
        fontsize=13
    )


    ax.set_xlim(
        0,
        1
    )

    ax.set_ylim(
        0,
        1
    )


    ax.grid(
        alpha=0.2
    )

    ax.legend(
        frameon=False
    )


    plt.tight_layout()


    safe_name = (
        outcome_name
        .lower()
        .replace(" ", "_")
    )


    png_path = (
        FIGURE_DIR
        / f"Figure_Risk_Concentration_{safe_name}.png"
    )

    svg_path = (
        FIGURE_DIR
        / f"Figure_Risk_Concentration_{safe_name}.svg"
    )


    plt.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight"
    )

    plt.savefig(
        svg_path,
        bbox_inches="tight"
    )


    plt.show()


    print("\nSaved:")
    print(png_path)

    print("Saved:")
    print(svg_path)


# ============================================================
# SAVE TABLES
# ============================================================

bootstrap_path = (
    TABLE_DIR
    / "Table_PrimaryModelB_Bootstrap_CIs.csv"
)

risk_path = (
    TABLE_DIR
    / "Table_PrimaryModelB_Risk_Concentration.csv"
)

capture_path = (
    TABLE_DIR
    / "Table_PrimaryModelB_Cumulative_Event_Capture.csv"
)


primary_bootstrap.to_csv(
    bootstrap_path,
    index=False
)

risk_concentration.to_csv(
    risk_path,
    index=False
)

capture_curves.to_csv(
    capture_path,
    index=False
)


print("\nSaved:")
print(bootstrap_path)

print("\nSaved:")
print(risk_path)

print("\nSaved:")
print(capture_path)


# ============================================================
# INTEGRITY CHECKS
# ============================================================

assert len(
    primary_bootstrap
) == 2

assert len(
    risk_concentration
) == 6

assert (
    risk_concentration[
        "Event_Capture"
    ]
    .between(
        0,
        1
    )
    .all()
)

assert (
    risk_concentration[
        "Risk_Lift"
    ]
    > 0
).all()


print(
    "\nBootstrap and risk-concentration integrity checks passed."
)

print("\nCell 11 complete.")

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path.cwd().parent
TABLE_DIR = BASE / "outputs" / "tables"

bootstrap_ci = pd.read_csv(
    TABLE_DIR / "Table_PrimaryModelB_Bootstrap_CIs.csv"
)

risk_conc = pd.read_csv(
    TABLE_DIR / "Table_PrimaryModelB_Risk_Concentration.csv"
)

event_capture = pd.read_csv(
    TABLE_DIR / "Table_PrimaryModelB_Cumulative_Event_Capture.csv"
)

print("=" * 90)
print("PRIMARY MODEL B — BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 90)
display(bootstrap_ci)

print("\n" + "=" * 90)
print("RISK CONCENTRATION")
print("=" * 90)
display(risk_conc)

print("\n" + "=" * 90)
print("CUMULATIVE EVENT CAPTURE")
print("=" * 90)
display(event_capture)

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path.cwd().parent
TABLE_DIR = BASE / "outputs" / "tables"

capture = pd.read_csv(
    TABLE_DIR / "Table_PrimaryModelB_Cumulative_Event_Capture.csv"
)

thresholds = [0.01, 0.05, 0.10, 0.20, 0.30, 0.50]

summary_rows = []

for outcome, g in capture.groupby("Outcome"):
    g = g.sort_values("Population_Fraction").reset_index(drop=True)

    for t in thresholds:
        idx = (g["Population_Fraction"] - t).abs().idxmin()

        summary_rows.append({
            "Outcome": outcome,
            "Top_Risk_Fraction": t,
            "Population_Fraction": g.loc[idx, "Population_Fraction"],
            "Cumulative_Event_Capture": g.loc[idx, "Cumulative_Event_Capture"]
        })

capture_summary = pd.DataFrame(summary_rows)

capture_summary["Top_Risk_Percent"] = (
    100 * capture_summary["Top_Risk_Fraction"]
)

capture_summary["Events_Captured_Percent"] = (
    100 * capture_summary["Cumulative_Event_Capture"]
)

print("=" * 90)
print("PRIMARY MODEL B — EVENT CONCENTRATION IN HIGHEST-RISK PATIENTS")
print("=" * 90)

display(
    capture_summary[
        [
            "Outcome",
            "Top_Risk_Percent",
            "Events_Captured_Percent"
        ]
    ].round(2)
)

In [ ]:
bootstrap_ci = pd.read_csv(
    TABLE_DIR / "Table_PrimaryModelB_Bootstrap_CIs.csv"
)

risk_conc = pd.read_csv(
    TABLE_DIR / "Table_PrimaryModelB_Risk_Concentration.csv"
)

print("\nBOOTSTRAP CONFIDENCE INTERVALS")
display(bootstrap_ci)

print("\nRISK CONCENTRATION")
display(risk_conc)